# EDA — Walmart Store Sales Forecasting

Joint notebook (both teammates). Findings feed the README:
- 3331 (Store, Dept) series, 143 weeks, 39-week test horizon
- holiday weeks weigh 5x in WMAE
- MarkDowns only exist after Nov 2011 (but fully present in test period)
- CPI/Unemployment missing in the test tail
- 1285 negative-sales rows (returns), 11 cold-start test series

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # repo root on path

import numpy as np
import pandas as pd
import mlflow

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow

train, test, features, stores = load_raw()
print(train.shape, test.shape)

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# Total weekly sales with holiday markers — the two December spikes rule this dataset
tot = train.groupby("Date")["Weekly_Sales"].sum()
hol = train.loc[train.IsHoliday, "Date"].unique()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(tot.index, tot.values)
for h in hol:
    ax.axvline(h, color="red", alpha=0.3, ls="--")
ax.set_title("Total weekly sales (red = holiday weeks)")
plt.show()

In [ ]:
# Seasonality: dept x week-of-year heatmap (top-25 depts by volume)
t = train.assign(woy=train.Date.dt.isocalendar().week.astype(int))
top = t.groupby("Dept").Weekly_Sales.sum().nlargest(25).index
pv = t[t.Dept.isin(top)].pivot_table(index="Dept", columns="woy", values="Weekly_Sales", aggfunc="mean")
pv = pv.div(pv.mean(axis=1), axis=0)  # normalize per dept
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pv, aspect="auto", cmap="RdBu_r", vmin=0, vmax=2)
ax.set_yticks(range(len(pv))); ax.set_yticklabels(pv.index)
ax.set_xlabel("week of year"); ax.set_title("Seasonal profile per dept (1 = dept average)")
fig.colorbar(im); plt.show()

In [ ]:
# Store types, series lengths, negatives
print(stores.groupby("Type").agg(n=("Store", "count"), avg_size=("Size", "mean")))
lens = train.groupby(["Store", "Dept"]).size()
print("\nseries length distribution:\n", lens.describe())
print("\nnegative sales rows:", (train.Weekly_Sales < 0).sum())
tr_series = set(train.groupby(["Store", "Dept"]).groups)
te_series = set(test.groupby(["Store", "Dept"]).groups)
print("cold-start test series (no train history):", len(te_series - tr_series))

In [ ]:
# Autocorrelation of a large series -> the lag-52 peak that motivates everything
from pandas.plotting import autocorrelation_plot
s = train[(train.Store == 20) & (train.Dept == 92)].set_index("Date").Weekly_Sales
fig, ax = plt.subplots(figsize=(10, 3))
autocorrelation_plot(s, ax=ax)
ax.set_title("Store 20 / Dept 92 — note the spike near lag 52")
plt.show()

In [ ]:
# Markdown availability over time (why NA -> 0 + presence flag)
f = features.set_index("Date")[[f"MarkDown{i}" for i in range(1, 6)]]
f.notna().groupby(level=0).mean().plot(figsize=(14, 3), title="Share of stores reporting each MarkDown")